# Clayton Metang — Expedition
High-level workflow using `expedition.py`. Each cell calls one stage; `x.save()` persists config after each step.

In [8]:
%load_ext autoreload
%autoreload 2
import logging
import claytonlib as clayton
from claytonlib.expedition import expedition

# --- Logging ---
# INFO shows per-write-cycle timing; DEBUG adds per-turn RNG details
logging.basicConfig(level=logging.INFO)
# logging.getLogger('claytonlib').setLevel(logging.INFO)

x = expedition("metang")
x.reload()
x.print()
x.chart_options.evaluation_frames_per_write_cycle = 5

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[expedition] Reloaded from data/expeditions/metang.json
=== Expedition: metang ===
  pokemon                  metang
  key_seed                 0x0C0E02C2
  setup_delay_s            180
  max_target_s             600
  strategy                 six-bait-then-balls
  criteria                 machete-50-turns-after-5-balls
  eval_strategy            sliding_window_13
  fps_model                linear
  window                   120
  target_delay             20058
  initial_time             2000-07-24T14:45:55
  target_seeds             ['0x1C1562D2']
  metronome_histsz         10
  metronome_second_window  2
  compass_m_delay          2441
  ---
  delay_from_key           19352 frames  (322.53s)


In [16]:
x.adjust(strategy_name="six-bait-then-balls")

[expedition] strategy_name = 'six-bait-then-balls'


# Chart - Finding a target

## Create chart

In [3]:
x.precompute_chart()
x.save()

[expedition] 14:46:12  === precompute_chart ===  (2026-09-11)
[expedition] 14:46:12  charting metang key_seed=0x0C0E02C2 delay=180-600s strategy=six-bait-then-balls criteria=machete-50-turns-after-5-balls  fps_models=['linear', 'quad'] (union)  workers=12
[expedition] 14:46:41  mdmsh 1/246  elapsed 0.5m  eta ~120.8m
[expedition] 14:48:33  mdmsh 5/246  elapsed 2.4m  eta ~113.3m
[expedition] 14:50:51  mdmsh 10/246  elapsed 4.6m  eta ~109.7m
[expedition] 14:53:10  mdmsh 15/246  elapsed 7.0m  eta ~107.4m
[expedition] 14:55:28  mdmsh 20/246  elapsed 9.3m  eta ~104.7m
[expedition] 14:57:48  mdmsh 25/246  elapsed 11.6m  eta ~102.6m
[expedition] 15:00:07  mdmsh 30/246  elapsed 13.9m  eta ~100.2m
[expedition] 15:02:17  mdmsh 35/246  elapsed 16.1m  eta ~96.9m
[expedition] 15:03:53  mdmsh 40/246  elapsed 17.7m  eta ~91.1m
[expedition] 15:05:54  mdmsh 45/246  elapsed 19.7m  eta ~88.0m
[expedition] 15:08:32  mdmsh 50/246  elapsed 22.3m  eta ~87.5m
[expedition] 15:10:45  mdmsh 55/246  elapsed 24.6m 

## Evaluate chart to find targets

`chart_report()` ranks the best **(boot time, commanded countdown M)** pairs across all candidate boot times (mode A), or the best M for a boot time you pass as `initial_time=` (mode B). It saves its ranked findings so `select_target()` can use them.

In [4]:
x.chart_report()
x.save()

[expedition] 16:35:41  === chart_report ===
Best (boot time, M) pairs  [top 10 of 7]  (jitter kernel, k=3.5):
   #            boot time     M (ms)  target F_b  second  P(capture)   sigma
   1  2000-07-27 14:53:26     268337       16494     274       27.1%    62.1
   2  2000-06-29 14:56:38     195202       12110     200       26.8%    52.9
   3  2000-07-29 14:57:08     463584       28198     469       26.6%    81.6
   4  2000-06-27 14:56:50     244581       15070     250       26.0%    59.3
   5  2000-05-30 14:59:59     401660       24486     407       26.0%    75.9
   6  2000-07-28 14:59:13     279614       17170     285       25.6%    63.4
   7  2000-06-27 14:47:59     327792       20058     333       25.0%    68.6
[expedition] 16:36:11  best target for each of 2464 starting times also saved
[expedition] 16:36:11  findings saved -> data/metang_0C0E02C2/chart_six-bait-then-balls_machete-50-turns-after-5-balls/chart_report.json  (top 7 + 2464 per-starting-time)
[expedition] Saved to dat

## Choose Target

`select_target()` reads the findings `chart_report()` saved and lets you pick one. It records the chosen **boot time** (`initial_time`), **timer countdown** (`target_timer_delay` = M), and **expected battle frame** (`target_delay` = F_b) on the expedition, then saves.

In [5]:
x.select_target()

[expedition] 16:36:26  === select_target ===



Select by  [t] top ranking   [s] specific starting time   [l] reuse last (07-24 14:45:55)  (blank to cancel):  l


  -> best target for 2000-07-24 14:45:55: M=327792 ms, F_b=20058, P~25.0%
[expedition] Saved to data/expeditions/metang.json
[expedition] 16:36:29  target set: boot 2000-07-24T14:45:55, timer M=327792 ms, expected F_b=20058 (P~25.0%). Saved.
[expedition] 16:36:29  predicted battle time (m/d h:m:s): 07-24 14:51:28  (= boot + 333s; year is the chart's 2000)


{'rank': 1753,
 'initial_time': '2000-07-24T14:45:55',
 'M': 327792,
 'target_delay': 20058,
 'second': 333,
 'p': 0.25010878830791183,
 'sigma': 68.60681354006711,
 'mdmsh': [247, 14]}

## Examine target area

In [7]:
 x.check().chart_check_target_landing()


chart_check_target_landing  (jitter kernel, k=3.5)
boot=2000-07-24T14:45:55  timer M=327792 ms  ->  mean F_b=20058.0 (target_delay=20058)  sigma=68.6
RTC-second distribution (σ_S=0.52s):  333=66%  334=20%  332=13%  335=0%  331=0%

=== second 333  P(S=333)=66.1%   mdmsh(m,h)=(247, 14)   battle 07-24 14:51:28
    cp(this second) = 27.63%   ->  contributes P·cp = 18.27% to the total
      frame      Δ        seed  hit    weight        w%      cumP%
    --------------------------------------------------------------
      19817   -241  0xF70E4D69    ✗    0.0021    0.001%     0.000%
      19818   -240  0xF70E4D6A    ✗    0.0022    0.001%     0.000%
      19819   -239  0xF70E4D6B    ✗    0.0023    0.001%     0.000%
      19820   -238  0xF70E4D6C    ✗    0.0024    0.001%     0.000%
      19821   -237  0xF70E4D6D    ✗    0.0026    0.001%     0.000%
      19822   -236  0xF70E4D6E    ✗    0.0027    0.002%     0.000%
      19823   -235  0xF70E4D6F    ✗    0.0028    0.002%     0.000%
      19824  

{'p': 0.2501060859352967,
 'seconds': [{'second': 333,
   'p_second': 0.6610705924920521,
   'cp': 0.2763340030862038},
  {'second': 334, 'p_second': 0.20281443767314977, 'cp': 0.21123908190858687},
  {'second': 332, 'p_second': 0.13200662971665478, 'cp': 0.180071713587139},
  {'second': 335, 'p_second': 0.002942416190694132, 'cp': 0.19364191629597008},
  {'second': 331,
   'p_second': 0.0011659239274491335,
   'cp': 0.2118766448254356}],
 'mismatches': None}

# Compass - Identify target

## Calibrate using metronome

In [ ]:
x.metronome_compass()
x.save()

## Finding what seed you hit in safari

`compass_safari()` builds candidates from the calibrated model: for the commanded countdown **M** (set by `select_target`) it sweeps the battle-frame window **F\* ± kσ** across second offsets **δ∈{−1,0,+1}** (off-by-one timer-start timing — each δ uses the *same* frame window). No hand-set delay window.

As you enter observed turns it ranks survivors by **posterior landing probability** (`P(land)`), shows the most-likely seed and which **δ** you hit ("timer on time / +1s late"), and flags when one candidate passes the confidence threshold. Extra commands:

- **`w`** — widen the frame (`k`) and/or second (`±K`) window and re-apply your path so far (also offered automatically on a no-match).
- The set is bounded to the seeds carrying `mass_cap` (default 0.999) of the landing probability; the Jane offload tip triggers on the *prior-weighted* effective count.

Pass `second_offsets=` / `mass_cap=` to override. Afterwards, `x.save_safari_run()` logs the identified seed, observed path, and inferred timer offset to `data/safari_runs.jsonl` (no capture required) for future model retuning.

In [ ]:
x.compass_safari()
x.save()

In [ ]:
# Loop-back: log this run (seed, observed path, inferred timer offset) for model retuning.
# No capture required — records even a fled/ambiguous run.
x.save_safari_run()

# Machete - Finding a path through seed

This is usually triggered during the "Finding what seed you hit in safari" step, but here's some manual activation anyways

## Finding a path for a single seed

In [ ]:
x.machete_one(max_turns=1000)
x.save()